In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('assets/Divar.csv')
df.head()

In [ ]:
df.drop(columns=['Unnamed: 0'], inplace=True)
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()
print(df.isnull().sum())

In [ ]:
# Calculate Missing Values as Percentage
# round((df.isnull().sum() / df.shape[0]) * 100, 2)
((df.isnull().sum() / len(df)) * 100).sort_values(ascending=True)

In [ ]:
df['cat3_slug'].unique()

In [ ]:
df['rooms_count'].unique()

### Preproccessing

In [ ]:
import pandas as pd
import numpy as np
import re

# ==========================================
# 1. Drop High Null Columns Function
# ==========================================
def drop_high_null_columns(df, threshold=97.1943):
    """
    Drops columns from the DataFrame that have a percentage of missing values 
    greater than the specified threshold.
    """
    df = df.copy()
    null_percentage = df.isnull().mean() * 100
    columns_to_drop = null_percentage[null_percentage > threshold].index.tolist()
    
    df.drop(columns=columns_to_drop, inplace=True)
    
    print(f"--- Drop High Null Columns ---")
    print(f"{len(columns_to_drop)} columns were dropped: {columns_to_drop}")
    return df

# ==========================================
# 2. Drop Unnecessary Columns Function (New)
# ==========================================
def drop_unnecessary_columns(df):
    """
    Drops specific columns that are no longer needed for analysis or modeling.
    """
    df = df.copy()
    columns_to_drop = ['rent_mode', 'credit_mode', 'floor_material']
    
    # Filter the list to ensure we only try to drop columns that actually exist
    existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
    
    df.drop(columns=existing_columns_to_drop, inplace=True)
    
    print(f"\n--- Drop Unnecessary Columns ---")
    print(f"Successfully dropped {len(existing_columns_to_drop)} columns: {existing_columns_to_drop}")
    return df

# ==========================================
# 3. Impute Boolean Amenities Function
# ==========================================
def impute_boolean_amenities(df):
    """
    Imputes missing boolean amenities using text mining on title and description.
    Maps existing string logic to integers.
    """
    df = df.copy()
    
    # Merge text columns for faster searching
    text_corpus = df['title'].fillna('') + ' ' + df['description'].fillna('')
    
    amenities_keywords = {
        'has_gas': ['گاز', 'فول امکانات', 'مبله', 'انشعابات کامل', 'انشعاب گاز'],
        'has_water': ['آب', 'فول امکانات', 'انشعابات کامل', 'انشعاب آب'],
        'has_electricity': ['برق', 'فول امکانات', 'انشعابات کامل', 'انشعاب برق'],
        'has_balcony': ['بالکن', 'تراس', 'بهارخواب', 'بهار خواب'],
        'has_elevator': ['آسانسور', 'اسانسور', 'لاین آسانسور'],
        'has_parking': ['پارکینگ', 'پارکینگ اختصاصی', 'پیکینگ'],
        'has_warehouse': ['انباری', 'انبار'],
        'has_pool': ['استخر', 'مجموعه آبی', 'استخردار'],
        'has_jacuzzi': ['جکوزی', 'جکوزی اختصاصی'],
        'has_sauna': ['سونا', 'سونا خشک', 'سونا بخار'],
        'has_barbecue': ['باربیکیو', 'باربیکیو', 'کباب پز', 'کباب‌پز'],
        'has_security_guard': ['نگهبان', 'نگهبانی', 'سرایدار', 'سرایداری', 'حفاظت', 'لابی من', 'لابی‌من'],
        'is_rebuilt': ['بازسازی', 'نوسازی', 'صفر تا صد', 'کلید نخورده', 'شیک']
    }

    print(f"\n--- Imputing Amenities ---")
    for col, keywords in amenities_keywords.items():
        if col in df.columns:
            # Map existing text values
            if df[col].dtype == 'object':
                df[col] = df[col].map({'دارد': 1, 'ندارد': 0, True: 1, False: 0})
            
            # Text mining for missing values
            null_mask = df[col].isnull()
            pattern = '|'.join(keywords)
            contains_keyword = text_corpus.str.contains(pattern, case=False, na=False)
            
            # Impute and cast to int
            df.loc[null_mask, col] = contains_keyword[null_mask].astype(int)
            df[col] = df[col].astype(int)
            
            print(f"Column '{col}' successfully processed.")
            
    return df

# ==========================================
# 4. Clean Standard Features Function
# ==========================================
persian_to_english = str.maketrans('۰۱۲۳۴۵۶۷۸۹', '0123456789')

def _clean_construction_year(year_val):
    if pd.isna(year_val):
        return np.nan
    year_str = str(year_val).strip()
    if year_str == 'قبل از ۱۳۷۰':
        return 1370.0 
    return float(year_str.translate(persian_to_english))

def _extract_year_from_description(text):
    if pd.isna(text):
        return np.nan
    text = str(text)
    if any(keyword in text for keyword in ['نوساز', 'کلید نخورده', 'کلیدنخورده', 'صفر']):
        return 1403.0
    pattern_forward = r'ساخت\s*([۱۳13|۱۴14][۰-۹0-9]{3})'
    pattern_backward = r'([۱۳13|۱۴14][۰-۹0-9]{3})\s*ساخت'
    match = re.search(pattern_forward, text) or re.search(pattern_backward, text)
    if match:
        year_str = match.group(1)
        return float(year_str.translate(persian_to_english))
    return np.nan

def clean_standard_features(df):
    """
    Cleans and standardizes categorical and numerical columns such as 
    rooms_count, construction_year, and price_mode.
    """
    df = df.copy()
    print(f"\n--- Cleaning Standard Features ---")
    
    # 1. Processing 'rooms_count'
    if 'rooms_count' in df.columns:
        rooms_mapping = {'بدون اتاق': 0, 'یک': 1, 'دو': 2, 'سه': 3, 'چهار': 4, 'پنج یا بیشتر': 5}
        df['rooms_count'] = df['rooms_count'].map(rooms_mapping)
        print("Column 'rooms_count' successfully mapped.")

    # 2. Processing 'construction_year'
    if 'construction_year' in df.columns:
        # Step A: Clean structured column
        df['construction_year'] = df['construction_year'].apply(_clean_construction_year)
        
        # Step B: Text-mining for missing values
        missing_mask = df['construction_year'].isnull()
        if missing_mask.sum() > 0 and 'description' in df.columns:
            extracted_years = df.loc[missing_mask, 'description'].apply(_extract_year_from_description)
            df['construction_year'] = df['construction_year'].fillna(extracted_years)
            
        # Step C: Boundary Enforcement
        df['construction_year'] = df['construction_year'].clip(lower=1364.0, upper=1403.0)
        print("Column 'construction_year' successfully cleaned, imputed, and clipped.")

    # 3. Processing 'price_mode'
    if 'price_mode' in df.columns:
        price_mode_mapping = {'مقطوع': 'fixed', 'توافقی': 'agreement', 'مجانی': 'free'}
        df['price_mode'] = df['price_mode'].map(price_mode_mapping)
        print("Column 'price_mode' successfully translated.")

    return df

# ==========================================
# 5. Main Pipeline Execution
# ==========================================
def run_preprocessing_pipeline(df):
    """
    Runs the complete preprocessing pipeline on the input DataFrame.
    """
    print("Starting Preprocessing Pipeline...\n")
    print(f"Initial shape: {df.shape}")
    
    # اجرای متوالی توابع
    df_cleaned = drop_high_null_columns(df)
    df_cleaned = drop_unnecessary_columns(df_cleaned)
    df_cleaned = impute_boolean_amenities(df_cleaned)
    df_cleaned = clean_standard_features(df_cleaned)
    
    print("\nPipeline execution finished successfully!")
    print(f"Final shape: {df_cleaned.shape}")
    
    return df_cleaned

In [ ]:
df = run_preprocessing_pipeline(df)

# Check the results
print(df['construction_year'].unique())

### Descriptive Analysis 4: Sale Price Distribution Across Property Categories

**Context & Objective:**
Understanding the distribution of property sale prices (`price_value`) across different level-3 real estate categories (`cat3_slug`) is crucial for grasping market dynamics. The primary objective of this analysis is to visualize and compare the spread, central tendency (median), and variance of sale prices for various property types (e.g., apartments, lands, villas, commercial spaces).

**Data Challenges & Cleaning Strategy:**
User-generated pricing data in real estate platforms inherently contains extreme outliers. These anomalies typically arise from two main sources:
1. **Real Market Variance:** Genuine high-value properties such as large-scale lands, luxury villas, or entire commercial complexes.
2. **User Input Errors:** Typographical errors (e.g., dummy values with repeating digits like `1111` or `9999999`), or entering prices in *Rials* instead of *Tomans* (resulting in massive 13 or 14-digit numbers).

To ensure an accurate and highly readable representation of the market, we apply a robust data cleaning pipeline:
* **Digit Frequency Analysis:** Identifying and removing unnatural repeating digits.
* **Currency Correction:** Mathematically converting 13 and 14-digit values from Rials to Tomans (dividing by 10) to rescue valuable data points instead of deleting them.

**Visualization Approach:**
We utilize a **Boxplot** to display the cleaned price distributions. To maintain a readable scale without distorting the reality of the market, we apply **Tukey’s IQR method** (using a $3 \times \text{IQR}$ multiplier). This scientifically filters out *extreme* input errors while preserving genuine high-end properties (shown as individual outlier dots). Furthermore, setting the boxplot whiskers to the 10th and 90th percentiles ensures a clear, comparative view of the core market across all categories.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker

# ==========================================
# 0. Initial Setup & Filtering
# ==========================================
# Assuming 'df' is your main dataframe. We only work with rows having a valid 'price_value'
df_sell = df.dropna(subset=['price_value']).copy()

# ==========================================
# 1. Initial Analysis of Extreme Prices
# ==========================================
print("\n" + "="*60)
print(" 1. Initial Analysis of Extreme 'price_value' by Category")
print("="*60)

# Calculate basic statistics for each level-3 category
price_analysis = df_sell.groupby('cat3_slug')['price_value'].agg(
    Total_Listings='count',
    Mean_Price='mean',
    Median_Price='median',
    Percentile_99=lambda x: np.percentile(x, 99),
    Max_Price='max'
).reset_index()

price_analysis = price_analysis.sort_values(by='Max_Price', ascending=False)

# Format to Billions for readability
cols_to_format = ['Mean_Price', 'Median_Price', 'Percentile_99', 'Max_Price']
for col in cols_to_format:
    price_analysis[col] = (price_analysis[col] / 1e9).round(2).astype(str) + " B"

print(price_analysis.head(10).to_string(index=False))

# ==========================================
# Helper Function for Digit Analysis Table
# ==========================================
def generate_digit_table(dataframe, description):
    """Calculates and prints the digit frequency table."""
    dataframe['price_str'] = dataframe['price_value'].astype('Int64').astype(str)
    # Ensure we only check numeric strings
    dataframe = dataframe[dataframe['price_str'].str.isnumeric()].copy()
    dataframe['digit_count'] = dataframe['price_str'].str.len()
    
    digit_analysis = dataframe.groupby('digit_count').size().reset_index(name='Frequency')
    total = len(dataframe)
    digit_analysis['Percentage (%)'] = (digit_analysis['Frequency'] / total * 100).round(4)
    
    def label_digit_meaning(digits):
        if digits <= 4: return "Clearly Fake / Dummy Values"
        elif digits in [5, 6, 7]: return "Suspicious / Rent Entered as Price"
        elif digits in [8, 9, 10, 11]: return "Normal Range (Millions to Billions)"
        else: return "Extreme Values (Over Trillions / Rial Errors)"
        
    digit_analysis['Category_Estimation'] = digit_analysis['digit_count'].apply(label_digit_meaning)
    digit_analysis = digit_analysis.sort_values(by='digit_count').reset_index(drop=True)
    
    print("\n" + "-"*60)
    print(f" {description} (Total: {total:,})")
    print("-"*60)
    print(digit_analysis.to_string(index=False))
    return dataframe

# ==========================================
# 2. Raw Digit Analysis
# ==========================================
print("\n" + "="*60)
print(" 2. Digit Length Analysis (Raw Data)")
print("="*60)
df_sell = generate_digit_table(df_sell, "Raw Price Digits Distribution")

# ==========================================
# 3. Remove Fake Prices (Repeating Digits)
# ==========================================
print("\n" + "="*60)
print(" 3. Cleaning Data: Removing Repeating/Fake Digits")
print("="*60)

# Identify repeating digits (e.g., '111', '9999', '1')
df_sell['is_repeating'] = df_sell['price_str'].apply(lambda x: len(set(x)) == 1)
dropped_count = df_sell['is_repeating'].sum()
df_sell = df_sell[~df_sell['is_repeating']].copy()

print(f"Removed {dropped_count:,} listings with fake repeating digits.")
df_sell = generate_digit_table(df_sell, "Digit Distribution AFTER Removing Fakes")

# ==========================================
# 4. Convert Rial to Toman (13 & 14 Digits)
# ==========================================
print("\n" + "="*60)
print(" 4. Cleaning Data: Rial to Toman Conversion")
print("="*60)

# Identify 13 or 14 digit numbers
rial_mask = df_sell['digit_count'].isin([13, 14])
rial_count = rial_mask.sum()

# Divide by 10
df_sell.loc[rial_mask, 'price_value'] = df_sell.loc[rial_mask, 'price_value'] // 10

print(f"Converted {rial_count:,} listings from Rial to Toman (divided by 10).")
df_cleaned = generate_digit_table(df_sell, "FINAL Digit Distribution AFTER Rial Conversion")

# ==========================================
# 5. Question 4: Removing Extreme Outliers & Boxplot Visualization
# ==========================================
print("\n" + "="*60)
print(" 5. Descriptive Question 4: IQR Outlier Filtering & Boxplot")
print("="*60)

# Function to remove only EXTREME outliers using Tukey's method (3x IQR) per category
def filter_extreme_outliers(df, cat_col, value_col):
    df_filtered = pd.DataFrame()
    
    # Calculate bounds for each category separately
    for category, group_data in df.groupby(cat_col):
        Q1 = group_data[value_col].quantile(0.25)
        Q3 = group_data[value_col].quantile(0.75)
        IQR = Q3 - Q1
        
        # Using 3.0 multiplier to target only EXTREME outliers, keeping natural market variance
        upper_bound = Q3 + (3.0 * IQR)
        
        # Keep valid data
        valid_data = group_data[group_data[value_col] <= upper_bound]
        df_filtered = pd.concat([df_filtered, valid_data])
        
    return df_filtered

# Apply the mathematical filter
initial_len = len(df_cleaned)
df_final = filter_extreme_outliers(df_cleaned, 'cat3_slug', 'price_value')
extreme_outliers_dropped = initial_len - len(df_final)

print(f"Dropped {extreme_outliers_dropped:,} extreme outliers (beyond 3x IQR) for a cleaner visualization.")
print("Natural market outliers (between 1.5x and 3x IQR) are kept and will be displayed as dots.")

# ------------------------------------------
# Plotting the Boxplot
# ------------------------------------------
print("Generating boxplot... Please check the plot window.")
plt.figure(figsize=(16, 8))

# Using the final filtered dataset
sns.boxplot(
    data=df_final, 
    x='cat3_slug', 
    y='price_value', 
    whis=1.5,            # Standard whisker length (1.5x IQR)
    showfliers=True,     # TURNED ON to show the remaining valid outliers
    palette='viridis',
    fliersize=4,         # Size of the outlier dots
    flierprops={'marker': 'o', 'markerfacecolor': 'orange', 'markeredgecolor': 'red', 'alpha': 0.4} # Styling outliers
)

# Formatting
plt.title('Distribution of Sale Price by Level-3 Category\n(Extreme Outliers Removed via 3x IQR Method)', 
          fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Level-3 Category (cat3_slug)', fontsize=12, fontweight='bold')
plt.ylabel('Price Value (Billions of Tomans)', fontsize=12, fontweight='bold')

plt.xticks(rotation=45, ha='right', fontsize=10)

def billions_formatter(x, pos):
    return f"{x / 1e9:.1f} B"

plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(billions_formatter))
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### Descriptive Analysis 5: Geographical Distribution and Spatial Analysis of Real Estate Listings

**Objective:**
The primary goal of this phase was to visualize the spatial distribution of Divar's real estate listings across Iran. This helps in identifying high-density housing hubs, regional property market concentrations, and urbanization patterns.

#### 1. Data Cleaning & Spatial Outlier Removal
Raw geographical data in real estate datasets often contains missing values and severe spatial outliers. Initial exploration revealed two main issues:
*   **Missing Coordinates (NaN):** A subset of property listings lacked exact latitude and longitude. In the real estate domain, this intentionally occurs due to **privacy concerns** (owners withholding exact addresses to prevent uncoordinated visits), **under-construction projects**, or **agency-level generic listings** where only the neighborhood name is provided instead of a precise map pinpoint.
*   **Spatial Outliers & Point-in-Polygon Filtering:** We identified records physically located in neighboring countries (e.g., Iraq, Gulf states) or oceans (0,0 coordinates) due to user VPNs, GPS glitches, or platform defaults. To strictly isolate the Iranian housing market, we utilized a **Point-in-Polygon** algorithm embedded with Iran's actual border coordinates. This successfully pruned the noise, keeping only valid domestic properties.

#### 2. Visualization Strategy: The Logarithmic Hexbin Heatmap
Visualizing the real estate distribution presented a "Skewed Data" challenge. Megacities, overwhelmingly led by Tehran, account for a massive volume of real estate transactions and construction activities compared to smaller provincial markets. A standard linear color scale would mask regional activities, highlighting only Tehran.

To solve this, we applied a **Logarithmic Scale (`bins='log'`)** to the Hexbin heatmap:
*   **Balancing the Scale:** The logarithmic approach dampens the extreme density of the capital, allowing mid-tier housing markets in other provinces to become visually distinct.
*   **Geographical Context:** We overlaid the continuous border of Iran and anchored major regional centers (e.g., Mashhad, Isfahan, Shiraz, Tabriz) onto the heatmap. This architectural layout translates abstract coordinates into a highly readable map of Iran's property market.

#### 3. Key Insights from the Housing Market
The refined heatmap clearly illustrates the macro-dynamics of Iran's real estate sector. While the **Tehran/Alborz metropolis** is the undisputed center of gravity for housing supply and real estate activities, the map reveals robust secondary property ecosystems. High-density real estate clusters are highly visible in the North-East (Khorasan), Central Iran (Isfahan), the South (Fars/Khuzestan), and the Caspian coastlines (Gilan/Mazandaran). This spatial distribution perfectly mirrors the country's urbanization trends, economic hubs, and areas with high housing demand.

In [ ]:
import folium
from folium.plugins import HeatMap
import pandas as pd

# 1. Clean geographical data
df_geo = df.dropna(subset=['location_latitude', 'location_longitude']).copy()
df_geo = df_geo[
    (df_geo['location_latitude'] >= 25) & (df_geo['location_latitude'] <= 40) &
    (df_geo['location_longitude'] >= 44) & (df_geo['location_longitude'] <= 63)
]

# 2. Create a base map centered on Iran
iran_map = folium.Map(location=[32.4279, 53.6880], zoom_start=5, tiles="CartoDB positron")

# 3. Prepare data points for HeatMap
heat_data = df_geo[['location_latitude', 'location_longitude']].values.tolist()

# 4. Customizing the HeatMap settings
custom_gradient = {
    0.2: 'blue',
    0.4: 'cyan',
    0.6: 'lime',
    0.8: 'yellow',
    0.95: 'orange',
    1.0: 'red'
}

HeatMap(
    heat_data, 
    radius=8, 
    blur=5,               
    max_zoom=12,             
    min_opacity=0.33,
    gradient=custom_gradient
).add_to(iran_map)

# 5. Save the interactive map as an HTML file and display it
iran_map.save("iran_real_estate_heatmap.html")
iran_map

In [ ]:
import os
import json
import urllib.request
import matplotlib.pyplot as plt
from matplotlib.path import Path
import pandas as pd
import numpy as np

# =========================================================
# 1. Automatically Download Iran's Border File
# =========================================================
geojson_filename = 'IRN.geo.json'
url = "https://raw.githubusercontent.com/johan/world.geo.json/master/countries/IRN.geo.json"

# Check if the file exists; if not, download it automatically
if not os.path.exists(geojson_filename):
    print("Downloading geographical border file for Iran... Please wait.")
    try:
        urllib.request.urlretrieve(url, geojson_filename)
        print("Download successful! File saved.")
    except Exception as e:
        print(f"Error during automatic download: {e}")
        print("Please ensure you have an active internet connection.")

# Load the downloaded GeoJSON file
with open(geojson_filename, 'r', encoding='utf-8') as f:
    iran_geojson = json.load(f)

# Extract polygons for the country borders and islands
iran_paths = []
for feature in iran_geojson['features']:
    geometry = feature['geometry']
    if geometry['type'] == 'Polygon':
        for subcoords in geometry['coordinates']:
            iran_paths.append(Path(subcoords))
    elif geometry['type'] == 'MultiPolygon':
        for polygon in geometry['coordinates']:
            for subcoords in polygon:
                iran_paths.append(Path(subcoords))

# =========================================================
# 2. Data Preparation and Exact Border Filtering
# =========================================================
# Drop missing coordinates
df_geo = df.dropna(subset=['location_latitude', 'location_longitude']).copy()
coords = df_geo[['location_longitude', 'location_latitude']].values

# Point in Polygon: Strictly filter out anything outside Iran (Iraq, Gulf, etc.)
print("Filtering data points to match strictly inside Iran's borders...")
is_inside_iran = np.zeros(len(coords), dtype=bool)
for path in iran_paths:
    is_inside_iran = is_inside_iran | path.contains_points(coords)

df_iran_only = df_geo[is_inside_iran]

print(f"Total valid geographical records: {len(df_geo):,}")
print(f"Records STRICTLY inside Iran borders: {len(df_iran_only):,}")

# =========================================================
# 3. Define Coordinates for Major Regional Reference Points
# =========================================================
regional_centers = {
    'Tehran / Alborz': (51.3892, 35.6892),
    'Mashhad': (59.6159, 36.2972),
    'Isfahan': (51.6660, 32.6546),
    'Shiraz': (52.5388, 29.5918),
    'Tabriz': (46.2919, 38.0962),
    'Ahvaz': (48.6706, 31.3183),
    'Rasht': (49.5831, 37.2808),
    'Kermanshah': (47.0778, 34.3142),
    'Kerman': (57.0740, 30.2839),
    'Zahedan': (60.8629, 29.4963)
}

# =========================================================
# 4. Plotting the Dashboard
# =========================================================
plt.figure(figsize=(14, 11))

# Layer 1: Draw the actual continuous border of Iran from the GeoJSON
for path in iran_paths:
    vertices = path.vertices
    plt.plot(vertices[:, 0], vertices[:, 1], color='black', linewidth=1.5, zorder=2)

# Layer 2: Plot the Hexbin Heatmap (Logarithmic Scale)
hb = plt.hexbin(
    df_iran_only['location_longitude'], 
    df_iran_only['location_latitude'], 
    gridsize=130,          
    cmap='YlOrRd', 
    bins='log',            
    mincnt=1, 
    edgecolors='none', 
    alpha=0.85,
    zorder=1
)

# Layer 3: Overlay City Markers and Names
for city, (lon, lat) in regional_centers.items():
    plt.scatter(lon, lat, color='black', marker='o', s=25, zorder=3)
    plt.text(
        lon + 0.15, lat + 0.15, city, 
        fontsize=9, fontweight='bold', color='black', zorder=4,
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=1.5)
    )

# =========================================================
# 5. Formatting and Customization
# =========================================================
cb = plt.colorbar(hb, orientation='vertical', fraction=0.03, pad=0.02)
cb.set_label('Density of Listings (Log10 Scale)', fontsize=12, fontweight='bold')

plt.title('Geographical Heatmap of Divar Listings\n(Strict Border Filtering + Regional Centers)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)

# Neatly clip the view window around Iran
plt.xlim(43.5, 64.0)
plt.ylim(24.5, 40.5)
plt.grid(True, linestyle='--', alpha=0.3, zorder=0)

plt.tight_layout()
plt.show()

### Hypothesis Test 3: The Impact of a Commercial Deed on Sale Price

**Context & Objective:**
In the real estate market, possessing a commercial deed (`has_business_deed`) signifies official and legal ownership, granting the owner full legal rights to the commercial property. The primary objective of this hypothesis test is to investigate whether having this official commercial deed has a statistically significant impact on the sale price of commercial properties.

To do this rigorously, we will perform three statistical checks:

**1. Normality Check (Shapiro-Wilk Test):**
*   **$H_0$ (Null):** The price data follows a normal distribution.
*   **$H_1$ (Alternative):** The price data does NOT follow a normal distribution.
*(This determines whether we use a parametric or non-parametric test).*

**2. Two-Sided Test (Mann-Whitney U):**
*   **$H_0$ (Null):** There is NO difference in the distribution of sale prices between properties with a deed and those without.
*   **$H_1$ (Alternative):** There IS a significant difference in the distribution of sale prices between the two groups.

**3. One-Sided Test (Mann-Whitney U - Greater):**
*   **$H_0$ (Null):** The sale prices of properties with a deed are less than or equal to those without a deed.
*   **$H_1$ (Alternative):** The sale prices of properties with a deed are strictly GREATER than those without a deed.

In [ ]:
### Hypothesist Testing - Q3

import scipy.stats as stats

# =====================================================
# Define Categories and Filter Valid Commercial Properties
# =====================================================
commercial_sale_cats = [
    'shop-sell', 
    'office-sell', 
    'industry-agriculture-business-sell'
]

# We filter prices > 0 to drop invalid negative or zero entries
df_comm = df[df['cat3_slug'].isin(commercial_sale_cats) & (df['price_value'] > 0)].copy()

# Process 'has_business_deed' (True/False -> 1/0)
df_comm['has_business_deed'] = df_comm['has_business_deed'].fillna(False).astype(int)

# Split the Data
group_with_deed = df_comm[df_comm['has_business_deed'] == 1]['price_value']
group_without_deed = df_comm[df_comm['has_business_deed'] == 0]['price_value']

# Calculate Medians and Mean for real-world interpretation
median_with = group_with_deed.median()
mean_with = group_with_deed.mean()

median_without = group_without_deed.median()
mean_without = group_without_deed.mean()

alpha = 0.05

# =====================================================
# STEP 1: NORMALITY CHECK (Shapiro-Wilk Test)
# =====================================================
print("=====================================================")
print(" STEP 1: NORMALITY CHECK (Shapiro-Wilk Test)")
print("=====================================================")
print("H0: The data follows a normal distribution.")
print("H1: The data does NOT follow a normal distribution.\n")

stat_shapiro_with, p_shapiro_with = stats.shapiro(group_with_deed)
stat_shapiro_without, p_shapiro_without = stats.shapiro(group_without_deed)

print(f"Group WITH Deed    -> Shapiro Statistic: {stat_shapiro_with:.4f}, P-value: {p_shapiro_with:.4f}")
print(f"Group WITHOUT Deed -> Shapiro Statistic: {stat_shapiro_without:.4f}, P-value: {p_shapiro_without:.4f}")

if p_shapiro_with < alpha and p_shapiro_without < alpha:
    print("\nConclusion: Both P-values are < 0.05. We REJECT the null hypothesis (H0).")
    print("Decision: The data is NOT normally distributed. Proceeding to non-parametric Mann-Whitney U test.")
else:
    print("\nConclusion: FAIL TO REJECT H0. Data might be normally distributed. Re-evaluate test choice.")

# =====================================================
# STEP 2: HYPOTHESIS TESTING (Two-Sided Mann-Whitney U)
# =====================================================
print("\n=====================================================")
print(" STEP 2: TWO-SIDED TEST (Mann-Whitney U)")
print("=====================================================")
print("H0: There is NO difference in price distributions between the two groups.")
print("H1: There IS a significant difference in price distributions.\n")

# Perform the two-sided Mann-Whitney U Test
u_stat_two, p_value_two = stats.mannwhitneyu(group_with_deed, group_without_deed, alternative='two-sided')

print(f"Sample size (With Deed): {len(group_with_deed):,} | Median Price: {median_with:,.0f}")
print(f"Sample size (Without Deed): {len(group_without_deed):,} | Median Price: {median_without:,.0f}")
print("-" * 55)
print(f"U-statistic: {u_stat_two:.2f}")
print(f"P-value: {p_value_two:.4f}")

if p_value_two < alpha:
    print("\nFinal Conclusion: REJECT the Null Hypothesis (H0).")
    print("Result: Accept H1. Having a business deed has a statistically significant impact on the commercial sale price.")
else:
    print("\nFinal Conclusion: FAIL TO REJECT the Null Hypothesis (H0).")

# =====================================================
# STEP 3: HYPOTHESIS TESTING (One-Sided Mann-Whitney U)
# =====================================================
print("\n=====================================================")
print(" STEP 3: ONE-SIDED TEST (Mann-Whitney U - Greater)")
print("=====================================================")
print("H0: Prices with a deed are <= prices without a deed.")
print("H1: Prices with a deed are strictly GREATER than prices without a deed.\n")

# Perform the one-sided Mann-Whitney U Test
u_stat_one, p_value_one = stats.mannwhitneyu(group_with_deed, group_without_deed, alternative='greater')

print(f"U-statistic: {u_stat_one:.2f}")
print(f"P-value (Greater): {p_value_one:.4f}")

if p_value_one < alpha:
    print("\nFinal Conclusion: REJECT the Null Hypothesis (H0).")
    print("Business Insight: Accept H1. Properties WITH a business deed are statistically proven to have HIGHER sale prices than those without.")
else:
    print("\nFinal Conclusion: FAIL TO REJECT the Null Hypothesis (H0).")
    print("Business Insight: We cannot prove that having a business deed makes the property more expensive.")
print("=====================================================")